In [1]:
import numpy as np
import cv2
# import imutils
import os
from os.path import join
import math
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

In [2]:
DATASET = './evs_mot_public_dataset'
TRAIN = os.path.join(DATASET, "evs_mot-train")
SEQUENCE = "MOT_05"

img = os.path.join(TRAIN, SEQUENCE, 'img1/%06d.jpg')
det_file = os.path.join(TRAIN, SEQUENCE, 'det', 'det.txt')
gt_file = os.path.join(TRAIN, SEQUENCE, 'gt', 'gt.txt')

In [5]:
def calculate_iou(boxA,boxB):
    boxA_x1, boxA_y1 = boxA[0], boxA[1]
    boxA_x2, boxA_y2 = boxA[0] + boxA[2], boxA[1] + boxA[3]
    boxB_x1, boxB_y1 = boxB[0], boxB[1]
    boxB_x2, boxB_y2 = boxB[0] + boxB[2], boxB[1] + boxB[3]

    xA = max(boxA_x1, boxB_x1)
    yA = max(boxA_y1, boxB_y1)
    xB = min(boxA_x2, boxB_x2)
    yB = min(boxA_y2, boxB_y2)

    width_iou = max(0, xB - xA)
    height_iou = max(0, yB - yA)
    interArea = width_iou * height_iou
    boxAreaA = boxA[2] * boxA[3]
    boxAreaB = boxB[2] * boxB[3]
    iou = interArea / (boxAreaA + boxAreaB - interArea)
    return iou

def calculate_centroid_distance(boxA, boxB):
    cx_A = boxA[0] + boxA[2] / 2
    cy_A = boxA[1] + boxA[3] / 2

    cx_B = boxB[0] + boxB[2] / 2
    cy_B = boxB[1] + boxB[3] / 2
    
    distance = ((cx_A - cx_B)**2 + (cy_A - cy_B)**2)**0.5
    
    return distance

In [ ]:
raw_det = np.loadtxt(det_file, delimiter=',')
detections_dict = {}

for row in raw_det:
    f = int(row[0])
    conf = row[6]
    if conf < 0.4:
        continue
    if f not in detections_dict:
        detections_dict[f] = []
    detections_dict[f].append(row[2:6])

active_tracks = {}
next_id = 1
lost_nr = 25
cap = cv2.VideoCapture(img, cv2.CAP_IMAGES)
frame_nr = 1

while True:
    ret, frame = cap.read()

    if not ret:
        break

    current_det = detections_dict.get(frame_nr, [])
    new_tracks = {}
    used_tracks = set()

    for det in current_det:
        best_id = None
        max_iou = 0.25

        for t_id, track in active_tracks.items():

            if t_id in used_tracks:
                continue

            t_bbox = track["bbox"]
            iou = calculate_iou(det, t_bbox)

            if iou > max_iou:

                max_iou = iou
                best_id = t_id

        if best_id is not None:
            new_tracks[best_id] = {"bbox": det, "lost" :0}
            used_tracks.add(best_id)

        else:
            new_tracks[next_id] = {"bbox": det, "lost" :0}
            next_id += 1

    for t_id, track in active_tracks.items():

        if t_id not in used_tracks:
            track["lost"] += 1

            if track["lost"] <= lost_nr:
                new_tracks[t_id] = track

    active_tracks = new_tracks
    for t_id, track in active_tracks.items():
        bbox = track["bbox"]

        if track["lost"] > 0:
            continue
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 0), 2)
        cv2.putText(frame, f'ID: {t_id}', (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

    cv2.imshow('Tracking', frame)
    frame_nr += 1
    if cv2.waitKey(15) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# ===== KALMAN TRACK CLASS =====
class KalmanTrack:
    def __init__(self, track_id, bbox):
        self.id = track_id
        self.bbox = np.array(bbox, dtype=np.float32)
        self.velocity = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)
        self.lost = 0
    
    def predict(self):
        return self.bbox + self.velocity
    
    def update(self, bbox):
        bbox = np.array(bbox, dtype=np.float32)
        self.velocity = (bbox - self.bbox) * 0.5 + self.velocity * 0.5
        self.bbox = bbox
        self.lost = 0

# ===== HUNGARIAN COST MATRIX =====
def compute_cost_matrix(active_tracks, detections, frame_width, frame_height):
    if not active_tracks or not detections:
        return np.array([]), list(active_tracks.keys())
    
    track_ids = list(active_tracks.keys())
    cost_matrix = np.zeros((len(track_ids), len(detections)))
    
    for i, track_id in enumerate(track_ids):
        predicted_bbox = active_tracks[track_id].predict()
        for j, det in enumerate(detections):
            iou = calculate_iou(predicted_bbox, det)
            dist = calculate_centroid_distance(predicted_bbox, det)
            norm_dist = dist / np.sqrt(frame_width**2 + frame_height**2)
            cost_matrix[i, j] = -0.7 * iou + 0.3 * norm_dist
    
    return cost_matrix, track_ids

raw_det = np.loadtxt(det_file, delimiter=',')
detections_dict = {}

for row in raw_det:
    f = int(row[0])
    conf = row[6]
    if conf < 0.4:
        continue
    if f not in detections_dict:
        detections_dict[f] = []
    detections_dict[f].append(row[2:6])

# ===== TRACKING =====
active_tracks = {}
next_id = 1
lost_nr = 25
cap = cv2.VideoCapture(img, cv2.CAP_IMAGES)
frame_nr = 1
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_det = detections_dict.get(frame_nr, [])
    
    # ===== HUNGARIAN MATCHING =====
    if active_tracks and current_det:
        cost_matrix, track_ids = compute_cost_matrix(active_tracks, current_det, frame_width, frame_height)
        track_indices, det_indices = linear_sum_assignment(cost_matrix)
        matched_pairs = set(track_ids[i] for i in track_indices)
    else:
        track_indices, det_indices = np.array([]), np.array([])
        track_ids = []
        matched_pairs = set()
    
    # ===== TE PĘTLE MUSZĄ BYĆ NA ZEWNĄTRZ if/else =====
    new_tracks = {}
    used_dets = set()
    
    # Update matched tracks
    for track_idx, det_idx in zip(track_indices, det_indices):
        track_id = track_ids[track_idx]
        active_tracks[track_id].update(current_det[det_idx])
        new_tracks[track_id] = active_tracks[track_id]
        used_dets.add(det_idx)
    
    # Unmatched detections = new tracks
    for det_idx, det in enumerate(current_det):
        if det_idx not in used_dets:
            new_tracks[next_id] = KalmanTrack(next_id, det)
            next_id += 1
    
    # Unmatched tracks = lost++
    for t_id, track in active_tracks.items():
        if t_id not in matched_pairs:
            track.lost += 1
            if track.lost <= lost_nr:
                new_tracks[t_id] = track
    
    active_tracks = new_tracks
    
    # DRAW
    for t_id, track in active_tracks.items():
        if track.lost == 0:
            x, y, w, h = map(int, track.bbox)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 0), 2)
            cv2.putText(frame, f'ID: {t_id}', (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

    cv2.imshow('Tracking', frame)
    frame_nr += 1
    if cv2.waitKey(15) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()

result_output_file = f"./results_{SEQUENCE}.txt"
with open(result_output_file, 'w') as f:
    f.writelines(results_list)

In [23]:
from collections import defaultdict

def load_ground_truth(gt_file):
    gt = defaultdict(dict)
    with open(gt_file, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            frame = int(parts[0])
            gt_id = int(parts[1])
            eval_flag = int(parts[6])
            obj_class = int(parts[7])
            
            if eval_flag != 1 or obj_class != 1:
                continue
            
            x, y, w, h = float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])
            gt[frame][gt_id] = [x, y, w, h]
    
    return gt

def calculate_iou_mota(boxA, boxB):
    x1_A, y1_A = boxA[0], boxA[1]
    x2_A, y2_A = boxA[0] + boxA[2], boxA[1] + boxA[3]
    
    x1_B, y1_B = boxB[0], boxB[1]
    x2_B, y2_B = boxB[0] + boxB[2], boxB[1] + boxB[3]
    
    xi1 = max(x1_A, x1_B)
    yi1 = max(y1_A, y1_B)
    xi2 = min(x2_A, x2_B)
    yi2 = min(y2_A, y2_B)
    
    inter_width = max(0, xi2 - xi1)
    inter_height = max(0, yi2 - yi1)
    inter_area = inter_width * inter_height
    
    area_A = boxA[2] * boxA[3]
    area_B = boxB[2] * boxB[3]
    
    union_area = area_A + area_B - inter_area
    iou = inter_area / union_area if union_area > 0 else 0
    return iou

gt_file = os.path.join(TRAIN, SEQUENCE, 'gt', 'gt.txt')
gt = load_ground_truth(gt_file)

results = defaultdict(dict)
with open(result_output_file, 'r') as f:
    for line in f:
        parts = line.strip().split(',')
        frame = int(parts[0])
        track_id = int(parts[1])
        x, y, w, h = float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])
        results[frame][track_id] = [x, y, w, h]

FN = 0
FP = 0
IDSW = 0
TP = 0
GT_total = 0

gt_to_track_history = {}

for frame in sorted(gt.keys()):
    gt_objects = gt[frame]
    tracked_objects = results.get(frame, {})
    
    GT_total += len(gt_objects)
    
    matched_tracks = set()
    
    for gt_id, gt_bbox in gt_objects.items():
        best_track_id = None
        best_iou = 0.5
        
        for track_id, track_bbox in tracked_objects.items():
            if track_id in matched_tracks:
                continue
            
            iou = calculate_iou_mota(gt_bbox, track_bbox)
            
            if iou > best_iou:
                best_iou = iou
                best_track_id = track_id
        
        if best_track_id is not None:
            TP += 1
            matched_tracks.add(best_track_id)
            
            if gt_id in gt_to_track_history:
                if gt_to_track_history[gt_id] != best_track_id:
                    IDSW += 1
            
            gt_to_track_history[gt_id] = best_track_id
        else:
            FN += 1
    
    FP += len(tracked_objects) - len(matched_tracks)

if GT_total > 0:
    MOTA = 1 - (FN + FP + IDSW) / GT_total
else:
    MOTA = 0

print(f"\nMOTA: {MOTA*100:.2f}%")
print(f"TP: {TP}, FN: {FN}, FP: {FP}, IDSW: {IDSW}, GT: {GT_total}")


MOTA: 44.69%
TP: 7533, FN: 5306, FP: 1421, IDSW: 374, GT: 12839
